# IDP

Nesse notebook, vamos explorar os recursos de Intelligent Document Processing da Landing AI.

A LandingAI é uma empresa fundada por **Andrew Ng** focada em aplicar Inteligência Artificial a problemas do mundo real, especialmente em Visão Computacional e processamento de documentos. Atualmente, um dos principais produtos da plataforma é o **Agentic Document Extraction (ADE)**, uma solução de **Document AI** voltada para transformar documentos em dados estruturados.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !pip install landingai-ade
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que usamos em outros notebooks, vamos utilizar mais algumas.
* `landingai_ade`: Biblioteca para trabalhar com os recursos da LandingAI.

In [ ]:
from getpass import getpass
from pprint import pprint
from pathlib import Path

from landingai_ade import LandingAIADE
from landingai_ade.types.v2.parse_response import V2ParseStructure

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image, Markdown

## 1. Obtendo Chave da API

Para conseguirem executar os trechos de código, vamos precisar de uma chave de API da LandingAI. O processo é relativamente simples.

### 1.1. Criar uma conta

Acesse o portal da LandingAI:

* [LandingAI](https://ade.landing.ai/)
* [Documentação ADE](https://docs.landing.ai/)

A plataforma oferece um ambiente chamado Playground, que permite testar o processamento de documentos diretamente pelo navegador antes mesmo de escrever código.

### 1.2. Localizar a API Key

Após criar a conta e acessar o portal:

1. Entrar no dashboard da [LandingAI](https://ade.landing.ai/).
2. Expandir o menu lateral esquerdo.
3. Localizar a seção de **API Keys**.
4. Gerar uma nova chave (**Create Key**).
5. Copiar e armazenar a chave com segurança.

A documentação oficial indica que as APIs ADE são acessadas por meio de uma API Key utilizada nas chamadas REST e nas bibliotecas cliente.

In [ ]:
API_KEY = getpass("Digite a API KEY copiada")

In [ ]:
client = LandingAIADE(apikey=API_KEY)

## 2. Processando Documento Simples

A solução da LandingAI possui diversos recursos. Vamos focar em duas funcionalidades:

* **Parse:** Responsável pelo OCR estruturado.
* **Extract:** Responsável para extração de dados estruturados a partir do OCR obtido pelo **parse**.

In [ ]:
img_bgr = cv2.imread('imagens/03/nota-fiscal.png')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

### 2.1. Parse do documento

Para processar o OCR estruturado do LandingAI, vamos executar o método `v2.parse()`.

In [ ]:
response = client.v2.parse(
    document=Path('imagens/03/nota-fiscal.png'),
    model='dpt-3-pro-latest',
    options={
        'atomic_grounding': False, # remove informações desnecessárias
    },
)

### 2.2. Entendendo a saída do Parse

As principais estruturas de saída do método `v2.parse()` são: `markdown`, `structure` e `metadata`.

#### 2.2.1. `markdown`

String com o conteúdo extraído em formato markdown.

In [ ]:
def salvar_markdown(
    markdown: str,
    arquivo: str,
    pasta_saida: str = 'output'
):
    pasta = Path(pasta_saida)

    # cria a pasta caso não exista
    pasta.mkdir(parents=True, exist_ok=True)

    caminho_arquivo = pasta / arquivo

    caminho_arquivo.write_text(
        markdown,
        encoding='utf-8'
    )

    return caminho_arquivo

In [ ]:
# Vamos salvar o markdown para o próximo notebook
salvar_markdown(response.markdown, 'nota-fiscal.md')

Markdown(response.markdown)

#### 2.2.2. `structure`

Possui a estrutura ou layout das informações obtidas.

In [ ]:
pprint(response.structure.to_dict(), sort_dicts=False)

In [ ]:
def apresentar_landingai_structure(
    image_rgb: np.ndarray,
    structure: V2ParseStructure,
    page_index: int = 0,
    draw_table_cells: bool = True,
    draw_labels: bool = True,
    thickness: int = 2
):
    COLORS = {
        'text':  (0, 255, 0),      # verde
        'table': (0, 120, 255),    # azul
        'logo':  (255, 0, 255),    # magenta
        'figure': (255, 255, 0),   # amarelo
        'default': (180, 180, 180)
    }

    img = image_rgb.copy()

    h, w = img.shape[:2]

    page = structure.children[page_index]

    for item in page.children:

        item_type = item.type

        if item_type == 'table_cell' and not draw_table_cells:
            continue

        box = item.grounding.box

        x1 = int(box.xmin * w)
        y1 = int(box.ymin * h)

        x2 = int(box.xmax * w)
        y2 = int(box.ymax * h)

        color = COLORS.get(
            item_type,
            COLORS['default']
        )

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            color,
            thickness
        )

        if draw_labels:

            label = item_type

            (tw, th), _ = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                1
            )

            cv2.rectangle(
                img,
                (x1, y1 - th - 8),
                (x1 + tw + 8, y1),
                color,
                -1
            )

            cv2.putText(
                img,
                label,
                (x1 + 4, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )

    plt.figure(figsize=(15, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
apresentar_landingai_structure(img_rgb, response.structure)

#### 2.2.3. `metadata`

Traz a estrutura dos metadados relacionados ao processamento do parse.

In [ ]:
pprint(response.metadata.to_dict(), sort_dicts=False)

### 2.3. Extração de Dados Estruturados

Vamos usar agora o método `v2.extract()` para extrair dados estruturados seguindo um **schema** que definirmos.

#### 2.3.1. Definindo o schema

In [ ]:
schema = {
    "type": "object",
    "properties": {

        "titulo_documento": {
            "type": "string",
            "description": "Título principal do documento"
        },

        "empresa": {
            "type": "object",
            "properties": {
                "nome": {
                    "type": "string"
                },
                
                "identificacao": {
                    "type": "string"
                }
            }
        },

        "informacoes_gerais": {
            "type": "object",
            "properties": {

                "numero_documento": {
                    "type": "string"
                },

                "data_emissao": {
                    "type": "string",
                    "description": "Formato yyyy-mm-dd"
                },

                "cliente": {
                    "type": "string"
                },

                "destinacao": {
                    "type": "string"
                }
            }
        },

        "itens": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {

                    "codigo": {
                        "type": "string"
                    },

                    "descricao": {
                        "type": "string"
                    },

                    "quantidade": {
                        "type": "number"
                    },

                    "valor_unitario": {
                        "type": "number"
                    },

                    "valor_total": {
                        "type": "number"
                    }

                }
            }
        },
    }
}

#### 2.3.2. Processando extração

Para processarmos a extração, precisaremos enviar o markdown obtido e o schema dos dados que queremos extrair.

In [ ]:
extract_response = client.v2.extract(
    markdown=response.markdown,
    schema=schema
)

### 2.4. Entendendo a saída do Extract

O retorno do método `v2.extract()` traz 2 estruturas legais de avaliar: `extraction` e `metadata`.

#### 2.4.1. `extraction`

Os valores extraídos de forma estruturada conforme schema.

In [ ]:
pprint(extract_response.extraction, sort_dicts=False)

#### 2.4.2. `metadata`

Metadados relacionados ao processamento de extração.

In [ ]:
pprint(extract_response.metadata.to_dict(), sort_dicts=False)

## 3. Processando Documento Complexo

Vamos olhar o comportamento do LandingAI em documentos complexos.

In [ ]:
img_bgr = cv2.imread('imagens/03/modelo-nfe.jpg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

### 3.1. Parse

In [ ]:
response = client.v2.parse(
    document=Path('imagens/03/modelo-nfe.jpg'),
    model='dpt-3-pro-latest',
    options={
        'atomic_grounding': False,
    },
)

In [ ]:
# Vamos salvar o markdown para o próximo notebook
salvar_markdown(response.markdown, 'modelo-nfe.md')

Markdown(response.markdown)

In [ ]:
apresentar_landingai_structure(img_rgb, response.structure)

### 3.2. Extract

In [ ]:
schema = {
    "type": "object",
    "properties": {

        "nota_fiscal": {
            "type": "object",
            "properties": {

                "numero": {
                    "type": "string",
                    "description": "Número da NF-e"
                },

                "serie": {
                    "type": "string"
                },

                "chave_acesso": {
                    "type": "string"
                },

                "data_emissao": {
                    "type": "string"
                }
            }
        },

        "emitente": {
            "type": "object",
            "properties": {

                "razao_social": {
                    "type": "string"
                },

                "cnpj": {
                    "type": "string"
                }
            }
        },

        "destinatario": {
            "type": "object",
            "properties": {

                "razao_social": {
                    "type": "string"
                },

                "cnpj": {
                    "type": "string"
                },

                "cidade": {
                    "type": "string"
                },

                "uf": {
                    "type": "string"
                }
            }
        },

        "itens": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {

                    "codigo": {
                        "type": "string"
                    },

                    "descricao": {
                        "type": "string"
                    },

                    "quantidade": {
                        "type": "number"
                    },

                    "valor_unitario": {
                        "type": "number"
                    },

                    "valor_total": {
                        "type": "number"
                    }
                }
            }
        },

        "totais": {
            "type": "object",
            "properties": {

                "valor_produtos": {
                    "type": "number"
                },

                "valor_frete": {
                    "type": "number"
                },

                "valor_nota": {
                    "type": "number"
                }
            }
        }
    }
}

In [ ]:
extract_response = client.v2.extract(
    markdown=response.markdown,
    schema=schema
)

In [ ]:
pprint(extract_response.extraction, sort_dicts=False)